# 0. Librerías y generalización de rutas

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import re, warnings, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
import seaborn as sns
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROC, FIG = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "reports" / "figuras"
for p in (RAW, PROC, FIG):
    p.mkdir(parents=True, exist_ok=True)

# Zonas con las que se entrena (texto que debe aparecer en la columna Línea). None = todas.
ZONAS = ["AutoNorte", "Calle 26", "NQS Sur"]

# Estaciones de estudio: código -> nombre corto. Los resultados se reportan para estas.
ESTUDIO = {"02200": "Alcalá", "06001": "Modelia", "07504": "Terreros"}

# Festivos del periodo (confirmar con un calendario oficial)
FESTIVOS = pd.to_datetime(["2026-06-08", "2026-06-15", "2026-06-29", "2026-07-13",
                           "2026-07-20", "2026-08-07", "2026-08-17"])

URL_GTFS = "https://storage.googleapis.com/gtfs-estaticos/GTFS_20260712.zip"
DIAS_TEST = 21        # los últimos días del periodo se reservan como conjunto de prueba
N_FOLDS = 5           # validación cruzada por grupos de fecha
RECARGAR = False      # True = volver a leer los Excel aunque exista el archivo procesado

In [ ]:
# Estilo común de las gráficas
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

AZUL, NARANJA, VERDE, GRIS, TINTA = "#2a78d6", "#eb6834", "#1baf7a", "#b9b8b2", "#52514e"
COLOR_DIA = {"habil": AZUL, "sabado": NARANJA, "domingo_festivo": VERDE}
NOMBRE_DIA = {"habil": "Día hábil", "sabado": "Sábado", "domingo_festivo": "Domingo o festivo"}
COLOR_EST = dict(zip(ESTUDIO.values(), [AZUL, NARANJA, VERDE]))
DIAS = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
MAPA_AZUL = LinearSegmentedColormap.from_list("azul", ["#fcfcfb", "#cde2fb", "#6da7ec", "#2a78d6", "#0d366b"])
MAPA_DIVERGENTE = LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRIS,
    "axes.grid": True, "grid.color": "#e7e6e2", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "lines.linewidth": 2, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": TINTA, "xtick.color": TINTA, "ytick.color": TINTA, "legend.frameon": False,
    "axes.prop_cycle": cycler(color=[AZUL, NARANJA, VERDE, "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"])})

def eje_horas(ax):
    ax.set_xticks(range(4, 24, 4)); ax.set_xticklabels([f"{h:02d}:00" for h in range(4, 24, 4)])
    ax.set_xlabel("Hora del día")

def miles(x, _=None):
    return f"{x / 1000:,.0f} mil".replace(",", ".") if abs(x) >= 1000 else f"{x:,.0f}"

# 1. Carga de Datos


## 2. Modelos Utilizados

### 2.1 Partición en entrenamiento y prueba

Se separa el 20 % de los registros para la prueba final, estratificando por estación: cada estación
queda con el mismo porcentaje de sus datos en entrenamiento y en prueba. Los datos de prueba no se
usan para ajustar ni para escoger modelos; solo para la comparación final de la sección 4.

In [ ]:
from sklearn.model_selection import train_test_split

X = data[["linea", "estación", "intervalo", "fecha", "tipo_dia"]]
y = data["validaciones"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=data["estación"], random_state=42)

print("Train:", X_train.shape, "| Test:", X_test.shape)
pct = X_test["estación"].value_counts() / data["estación"].value_counts() * 100
print(f"Estaciones en test: {X_test['estación'].nunique()} de {data['estación'].nunique()}"
      f" | % de cada estación en test: de {pct.min():.1f} a {pct.max():.1f}")

### 2.2 Preparación de las variables

- **Variable a predecir:** validaciones en la franja de 15 minutos.
- **Categóricas** (`linea`, `estación`, `tipo_dia`): codificación one-hot.
- **Hora:** el intervalo convertido a número (05:15 → 5,25) y estandarizado, como exigen Ridge, Lasso y los kernels.
- **Fecha:** no entra directamente; su información está en `tipo_dia`.

La preparación va dentro de un *pipeline*, de modo que en cada pliegue se ajusta solo con los datos
de entrenamiento de ese pliegue (sin fuga de información).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.kernel_approximation import Nystroem
from sklearn.model_selection import KFold, cross_validate, GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def a_hora(v):                       # "05:15" -> 5.25
    h, m = str(v).split(":")[:2]
    return int(h[-2:]) + int(m) / 60

X_train, X_test = X_train.copy(), X_test.copy()
X_train["hora"] = X_train["intervalo"].map(a_hora)
X_test["hora"] = X_test["intervalo"].map(a_hora)

CATEGORICAS, NUMERICAS = ["linea", "estación", "tipo_dia"], ["hora"]

def preparar():
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
                              ("num", StandardScaler(), NUMERICAS)])

con_splines = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
                                 ("spl", SplineTransformer(), NUMERICAS)])

cv = KFold(n_splits=5, shuffle=True, random_state=42)     # los mismos 5 pliegues para todos los modelos

### 2.3 Los seis modelos con sus parámetros por defecto

Primera comparación: cada método tal como viene, con validación cruzada de 5 pliegues **sobre
entrenamiento**. La línea base predice siempre la media y es el piso que todos deben superar.

| Modelo | Qué hace con estos datos |
|---|---|
| Base (media) | Predice el promedio de validaciones para todo |
| OLS | Un efecto por línea, estación y tipo de día, y una recta para la hora |
| Ridge | Igual que OLS, encogiendo los coeficientes |
| Lasso | Igual que OLS, pudiendo llevar coeficientes a cero |
| Splines | Reemplaza la recta de la hora por una curva suave |
| Kernel (RBF) | Parecido entre registros; aproximado con Nyström porque el kernel exacto no cabe en memoria |

In [ ]:
modelos = {"Base (media)": make_pipeline(preparar(), DummyRegressor()),
           "OLS":          make_pipeline(preparar(), LinearRegression()),
           "Ridge":        make_pipeline(preparar(), Ridge()),
           "Lasso":        make_pipeline(preparar(), Lasso()),
           "Splines":      make_pipeline(con_splines, LinearRegression()),
           "Kernel (RBF)": make_pipeline(preparar(), Nystroem(random_state=42), Ridge())}

filas = {}
for nombre, m in modelos.items():
    r = cross_validate(m, X_train, y_train, cv=cv, n_jobs=-1,
                       scoring=("neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"))
    filas[nombre] = {"MAE": -r["test_neg_mean_absolute_error"].mean(),
                     "RMSE": -r["test_neg_root_mean_squared_error"].mean(),
                     "RMSE (desv. entre pliegues)": r["test_neg_root_mean_squared_error"].std(),
                     "R2": r["test_r2"].mean()}
tabla_cv = pd.DataFrame(filas).T.sort_values("RMSE")
display(tabla_cv.round(3))

### 2.3 Búsqueda en grilla de los tres finalistas

Los tres modelos de menor RMSE en la tabla anterior pasan a búsqueda en grilla, con los mismos
pliegues y solo con datos de entrenamiento. Si un valor ganador queda en el borde de su grilla,
la celda lo avisa y la grilla se amplía hacia ese lado.

In [ ]:
GRILLAS = {"OLS":          {},
           "Ridge":        {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
           "Lasso":        {"lasso__alpha": [0.001, 0.01, 0.1, 1, 10]},
           "Splines":      {"columntransformer__spl__n_knots": [5, 10, 20, 40, 60, 80]},
           "Kernel (RBF)": {"nystroem__gamma": [0.03, 0.1, 0.3, 1],
                            "nystroem__n_components": [200],
                            "ridge__alpha": [0.001, 0.01, 0.1, 1, 10]}}

finalistas = tabla_cv.drop(index="Base (media)").index[:3].tolist()
print("Finalistas (los tres de menor RMSE en validación cruzada):", finalistas)

ajustados, filas = {}, {}
for nombre in finalistas:
    b = GridSearchCV(modelos[nombre], GRILLAS[nombre], cv=cv, n_jobs=-1,
                     scoring="neg_root_mean_squared_error").fit(X_train, y_train)
    ajustados[nombre] = b.best_estimator_
    filas[nombre] = {"RMSE en CV (por defecto)": tabla_cv.loc[nombre, "RMSE"],
                     "RMSE en CV (ajustado)": -b.best_score_,
                     "Hiperparámetros": ", ".join(f"{k.split('__')[-1]} = {v}" for k, v in b.best_params_.items()) or "no tiene"}
    for k, v in b.best_params_.items():
        if len(GRILLAS[nombre][k]) > 1 and v in (GRILLAS[nombre][k][0], GRILLAS[nombre][k][-1]):
            print(f"   OJO {nombre}: {k.split('__')[-1]} = {v} quedó en el borde de la grilla; ampliarla hacia ese lado")

comparacion = pd.DataFrame(filas).T.sort_values("RMSE en CV (ajustado)")
display(comparacion)
mejor = comparacion.index[0]
print("Mejor por validación cruzada:", mejor)

### 3.4 Matriz de aplicabilidad

| Método | ¿Aporta? | RMSE en CV | Justificación |
|---|---|---|---|
| Base (media) | Referencia | ___ | Piso de comparación |
| OLS | ___ | ___ | ___ |
| Ridge | ___ | ___ | ___ |
| Lasso | ___ | ___ | ___ |
| Splines | ___ | ___ | ___ |
| Kernel (RBF) | ___ | ___ | ___ |

**Finalistas:** ___, ___ y ___. **Hiperparámetros escogidos:** ___.

## 3. Comparación final en los datos de prueba

### 3.1 Error de los finalistas en prueba

Es la primera y única vez que se usan los datos de prueba. Los errores están en validaciones por
franja de 15 minutos.

In [ ]:
en_prueba = {"Base (media)": modelos["Base (media)"].fit(X_train, y_train), **{n: ajustados[n] for n in comparacion.index}}
pred = {nombre: m.predict(X_test) for nombre, m in en_prueba.items()}

tabla_test = pd.DataFrame({nombre: {"MAE": mean_absolute_error(y_test, p),
                                    "RMSE": root_mean_squared_error(y_test, p),
                                    "R2": r2_score(y_test, p)} for nombre, p in pred.items()}).T
tabla_test["Mejora en MAE frente a la base"] = (1 - tabla_test["MAE"] / tabla_test.loc["Base (media)", "MAE"]).map("{:.0%}".format)
display(tabla_test.round(3))

### 3.2 Bootstrap: ¿la diferencia entre finalistas es real?

Se remuestrean 1.000 veces los días de los datos de prueba (días completos, porque las franjas de un
mismo día se parecen entre sí) y se recalcula el error de cada modelo. Si el intervalo del 95 % de la
diferencia frente al mejor no incluye el 0, la ventaja es real.

In [ ]:
B = 1000
rng = np.random.default_rng(42)
dia = pd.to_datetime(X_test["fecha"]).dt.date.to_numpy()
dias = np.unique(dia)
muestras = rng.integers(0, len(dias), size=(B, len(dias)))          # los mismos días para todos los modelos

def por_dia(valores):                                               # suma de cada día, en el orden de `dias`
    return pd.Series(valores).groupby(dia).sum().reindex(dias).to_numpy()

n = por_dia(np.ones(len(X_test)))
boot = {}
for nombre in comparacion.index:                                    # del mejor al peor por CV
    e = y_test.to_numpy() - pred[nombre]
    abs_dia, cuad_dia = por_dia(np.abs(e)), por_dia(e ** 2)
    boot[nombre] = {"MAE": abs_dia[muestras].sum(1) / n[muestras].sum(1),
                    "RMSE": np.sqrt(cuad_dia[muestras].sum(1) / n[muestras].sum(1))}

def intervalo(v):
    return f"{v.mean():.2f}  [{np.percentile(v, 2.5):.2f}, {np.percentile(v, 97.5):.2f}]"

display(pd.DataFrame({k: {met: intervalo(v) for met, v in d.items()} for k, d in boot.items()}).T
        .rename(columns=lambda c: c + " en prueba, IC 95 %"))

for otro in comparacion.index[1:]:
    for met in ("MAE", "RMSE"):
        dif = boot[otro][met] - boot[mejor][met]
        lo, hi = np.percentile(dif, [2.5, 97.5])
        print(f"{met}: {otro} − {mejor} = {dif.mean():.2f}  [{lo:.2f}, {hi:.2f}]  ->",
              "diferencia real" if lo > 0 or hi < 0 else "no se distinguen (el intervalo incluye 0)")

### 3.3 Comportamiento en Alcalá, Modelia y Terreros

El mejor modelo frente a los datos reales en un lunes hábil de agosto escogido al azar. La línea es
la predicción; los puntos negros son datos de prueba y los grises son datos de entrenamiento de ese
mismo día. La tabla resume el error de cada estación en todos sus datos de prueba.

In [ ]:
modelo = ajustados[mejor]
ESTUDIO = ["Alcalá", "Modelia", "Terreros"]

fechas = pd.to_datetime(data["fecha"])
lunes = data[(fechas.dt.month == 8) & (fechas.dt.dayofweek == 0)]
lunes = lunes[lunes["tipo_dia"] == lunes["tipo_dia"].mode()[0]]        # quita el lunes festivo
FECHA = str(pd.to_datetime(lunes["fecha"]).drop_duplicates().sample(1, random_state=42).iloc[0].date())

fig, ejes = plt.subplots(1, 3, figsize=(13.5, 3.9))
errores = {}
for ax, est in zip(ejes, ESTUDIO):
    nombre = data.loc[data["estación"].str.contains(est, case=False), "estación"].iloc[0]
    d = data[(data["estación"] == nombre) & (fechas == FECHA)].copy()
    d["hora"] = d["intervalo"].map(a_hora)
    d = d.sort_values("hora")
    d["pred"] = modelo.predict(d[X_train.columns])
    en_test = d.index.isin(X_test.index)
    ax.scatter(d.hora[~en_test], d.validaciones[~en_test], s=14, color="#b9b8b2", label="Real (dato de entrenamiento)")
    ax.scatter(d.hora[en_test], d.validaciones[en_test], s=18, color="#1f1f1d", label="Real (dato de prueba)")
    ax.plot(d.hora, d.pred, color="#2a78d6", linewidth=2, label=mejor)
    ax.set_title(f"{est}, lunes {FECHA}", loc="left", fontweight="bold", fontsize=11)
    ax.set_xticks(range(4, 24, 4)); ax.set_xticklabels([f"{h:02d}:00" for h in range(4, 24, 4)])
    ax.set_xlabel("Hora del día")
    ax.spines[["top", "right"]].set_visible(False)

    t = X_test[X_test["estación"] == nombre]              # todas las fechas, solo datos de prueba
    p = modelo.predict(t)
    errores[est] = {"Registros en prueba": len(t), "Entradas promedio": y_test[t.index].mean(),
                    "MAE": mean_absolute_error(y_test[t.index], p),
                    "RMSE": root_mean_squared_error(y_test[t.index], p)}
ejes[0].set_ylabel("Entradas en la franja de 15 min")
fig.legend(*ejes[0].get_legend_handles_labels(), loc="lower center", ncol=3, frameon=False)
plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.show()

errores = pd.DataFrame(errores).T
errores["MAE / promedio"] = errores["MAE"] / errores["Entradas promedio"]
print("Modelo:", mejor, "| Error en los datos de prueba, todas las fechas:")
display(errores.round(2))

### 4.4 Decisión

- **Modelo escogido:** ___, con ___ (hiperparámetros).
- **Error en prueba:** MAE de ___ validaciones por franja (IC 95 %: ___ a ___), ___ % menos que la línea base.
- **Frente a los otros finalistas:** la diferencia con ___ es de ___ [___, ___] y con ___ de ___ [___, ___].
- **Por estación:** Alcalá ___, Modelia ___, Terreros ___ (MAE). Donde más falla es en ___, a las ___.
- **Limitaciones:** un mismo día tiene registros en entrenamiento y en prueba, así que el error mide
  qué tan bien se completan franjas de días ya vistos, no qué tan bien se predice un día nuevo;
  el modelo no usa la oferta de buses ni eventos (marchas, cierres); solo cubre tres troncales.